GOLD LAYER

1. The Master Join (The "Single Version of Truth")
First, we create the comprehensive table that powers most of your reports.

In [0]:
from pyspark.sql import functions as F

# Load cleaned DataFrames from silver Delta tables
cleaned_sales_transactions = spark.table("capstone_catalog.silver.cleaned_sales_transactions")
cleaned_product_master = spark.table("capstone_catalog.silver.cleaned_product_master")
cleaned_customer_data = spark.table("capstone_catalog.silver.cleaned_customer_data")
cleaned_store_master = spark.table("capstone_catalog.silver.cleaned_store_master")
cleaned_inventory_data = spark.table("capstone_catalog.silver.cleaned_inventory_data")
cleaned_clickstream_events = spark.table("capstone_catalog.silver.cleaned_clickstream_events")

# Create the Master Gold Table
gold_sales = cleaned_sales_transactions \
    .join(cleaned_product_master, "product_id", "left") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_store_master, "store_id", "left")

# Add a Profit Column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

In [0]:
# 1. First, create your lookup table
avg_selling_price_per_category = cleaned_product_master.groupBy("category") \
    .agg(F.avg("selling_price").alias("avg_category_selling_price"))

# 2. Join with Sales (Safe Version)
sales_with_prices = cleaned_sales_transactions.join(
    cleaned_product_master.select("product_id", "selling_price", "category"), 
    on="product_id",
    how="left"
).withColumnRenamed("selling_price", "master_selling_price")

# 3. Join with Average Price - USE STRING JOIN TO AVOID DUPLICATES
# By using on="category" (the string), Spark automatically keeps only ONE column named 'category'
sales_with_customer_info = sales_with_prices.join(
    avg_selling_price_per_category,
    on="category", 
    how="left"
)

# 4. Master Gold Join (If you are getting the error here)
# Make sure cleaned_sales_transactions DOES NOT already have category before this join
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, "product_id", "left") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_store_master, "store_id", "left") \
    .withColumn(
        "profit",
        F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
    )

from pyspark.sql import functions as F

# =====================================================
# MASTER GOLD TABLE — "Single Version of Truth"
# Joins: Sales + Products + Customers + Stores
# Handles duplicate columns (category, loyalty_status, city)
# =====================================================

gold_sales = cleaned_sales_transactions \
    .drop("category", "loyalty_status") \
    .join(cleaned_product_master, "product_id", "left") \
    .join(
        cleaned_customer_data.withColumnRenamed("city", "customer_city"),
        "customer_id", "left"
    ) \
    .join(cleaned_store_master, "store_id", "left") \
    .withColumn(
        "profit",
        F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
    )

print("=== MASTER GOLD TABLE SCHEMA ===")
gold_sales.printSchema()
print(f"\nTotal Records: {gold_sales.count():,}")
print("\nSample — 'A Gold customer in Mumbai Mall bought an Apple iPhone via Mobile App':")
gold_sales.select(
    "transaction_id", "order_date", "channel", "loyalty_status",
    "city", "store_type", "brand", "product_name", "category",
    "total_amount", "profit"
).show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

# 1. Join with Product Master
# We drop 'category' from the sales side BEFORE joining so we only keep the one from Product Master
sales_with_products = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left")

# 2. Complete the Master Join
# Using the string format on="column_id" automatically merges duplicate join keys
gold_sales = sales_with_products \
    .join(cleaned_customer_data, on="customer_id", how="left") \
    .join(cleaned_store_master, on="store_id", how="left")

# 3. Add the 'profit' column (solves your previous error too)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Check: Do we have duplicate categories?")
print([c for c in gold_sales.columns if c == "category"]) # Should print ['category'] once

In [0]:
from pyspark.sql import functions as F

# 1. Prepare Customer and Store data by renaming ambiguous columns
# This prevents 'city' and 'state' from appearing twice
customers_prepped = cleaned_customer_data \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("state", "customer_state")

stores_prepped = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create gold_sales with a clean join
# We drop 'category' from transactions to use the one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(customers_prepped, on="customer_id", how="left") \
    .join(stores_prepped, on="store_id", how="left")

# 3. Add the profit column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Gold Sales created. New city columns: 'customer_city' and 'store_city'")

In [0]:
# A. Top Performing Categories
category_performance = gold_sales.groupBy("category") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.count("transaction_id").alias("transaction_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_profit").desc())

print("=== TOP CATEGORIES BY PROFIT ===")
category_performance.show(truncate=False)



In [0]:
# B. Store Revenue by Store City (Updated Column Name)
city_store_performance = gold_sales.groupBy("store_city", "store_state", "store_type") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.countDistinct("transaction_id").alias("total_transactions")
    ).orderBy(F.col("total_revenue").desc())

print("=== REVENUE BY STORE CITY & TYPE ===")
city_store_performance.show(30, truncate=False)

### 3. Conversion Rate Analysis

Calculates the **Marketing Funnel**: View → Add-to-Cart → Purchase conversion rates from clickstream data. Excludes time-travel anomalies for accurate funnel metrics. Also breaks down conversion by platform.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CONVERSION RATE: (Total Purchases / Total Views)
# Excludes time-travel anomalies for accurate funnel
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Event distribution
print("=== EVENT TYPE DISTRIBUTION ===")
clean_clickstream.groupBy("event_type").count().orderBy("count").show()

total_views = clean_clickstream.filter(F.col("event_type") == "view").count()
total_add_to_cart = clean_clickstream.filter(F.col("event_type") == "add_to_cart").count()
total_purchases = clean_clickstream.filter(F.col("event_type") == "purchase").count()

overall_conversion = round((total_purchases / total_views) * 100, 2) if total_views > 0 else 0
view_to_cart = round((total_add_to_cart / total_views) * 100, 2) if total_views > 0 else 0
cart_to_purchase = round((total_purchases / total_add_to_cart) * 100, 2) if total_add_to_cart > 0 else 0

print("=== CONVERSION FUNNEL ===")
print(f"Total Views:         {total_views:,}")
print(f"Total Add-to-Cart:   {total_add_to_cart:,}")
print(f"Total Purchases:     {total_purchases:,}")
print(f"\nView → Purchase Rate:     {overall_conversion}%")
print(f"View → Add-to-Cart Rate:  {view_to_cart}%")
print(f"Cart → Purchase Rate:     {cart_to_purchase}%")

# Conversion Rate by Platform
conversion_by_platform = clean_clickstream.groupBy("platform") \
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("add_to_carts"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
    ).withColumn("conversion_rate_pct", F.round((F.col("purchases") / F.col("views")) * 100, 2))

print("\n=== CONVERSION RATE BY PLATFORM ===")
conversion_by_platform.orderBy(F.col("conversion_rate_pct").desc()).show(truncate=False)

### 4. Stock-out Risk Identification

Identifies products where `stock_on_hand < reorder_level` — these are at immediate risk of running out of stock. Enriched with product and store details for actionable insights.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# STOCK-OUT RISK: stock_on_hand < reorder_level
# =====================================================

stockout_risk = cleaned_inventory_data \
    .filter(F.col("stock_on_hand") < F.col("reorder_level")) \
    .join(cleaned_product_master.select("product_id", "product_name", "brand", "category"), "product_id", "left") \
    .join(cleaned_store_master.select("store_id", "store_name", "city"), "store_id", "left") \
    .select(
        "product_id", "product_name", "brand", "category",
        "store_id", "store_name", "city",
        "stock_on_hand", "reorder_level",
        (F.col("reorder_level") - F.col("stock_on_hand")).alias("units_below_threshold")
    ).orderBy(F.col("units_below_threshold").desc())

stockout_count = stockout_risk.count()
print(f"=== STOCK-OUT RISK: {stockout_count} product-store combinations at risk ===")
stockout_risk.show(20, truncate=False)

# Summary by category
stockout_by_category = stockout_risk.groupBy("category") \
    .agg(
        F.count("*").alias("at_risk_count"),
        F.sum("units_below_threshold").alias("total_units_deficit")
    ).orderBy(F.col("at_risk_count").desc())

print("=== STOCK-OUT RISK BY CATEGORY ===")
stockout_by_category.show(truncate=False)

# Summary by city
stockout_by_city = stockout_risk.groupBy("city") \
    .agg(F.count("*").alias("at_risk_count")) \
    .orderBy(F.col("at_risk_count").desc())

print("=== STOCK-OUT RISK BY CITY ===")
stockout_by_city.show(truncate=False)

### 5. Churn Risk Analysis

Identifies customers who **haven't purchased in the last 90 days** but **were active on the app** (clickstream). Uses data-relative dates (max dates from actual data) as reference points.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CHURN RISK: No purchase in 90 days but active on app
# Uses data-relative dates as reference
# =====================================================

# Determine reference dates from data
reference_purchase_date = cleaned_sales_transactions.agg(F.max("order_date")).collect()[0][0]
reference_activity_date = cleaned_clickstream_events.agg(F.max("event_timestamp")).collect()[0][0]

print(f"Reference Date (latest purchase in data): {reference_purchase_date}")
print(f"Reference Date (latest app activity):     {reference_activity_date}")

# Last purchase date per customer
last_purchase = cleaned_sales_transactions \
    .groupBy("customer_id") \
    .agg(F.max("order_date").alias("last_purchase_date"))

# Last clickstream activity per customer (excluding anomalies)
last_activity = cleaned_clickstream_events \
    .filter(F.col("is_time_travel_anomaly") == False) \
    .groupBy("customer_id") \
    .agg(F.max("event_timestamp").alias("last_app_activity"))

# Churn risk: no purchase in 90 days but active on app within 90 days
churn_risk = last_purchase.join(last_activity, "customer_id", "inner") \
    .withColumn("days_since_purchase", F.datediff(F.lit(reference_purchase_date), F.col("last_purchase_date"))) \
    .withColumn("days_since_app_activity", F.datediff(F.lit(reference_purchase_date), F.col("last_app_activity").cast("date"))) \
    .filter(
        (F.col("days_since_purchase") > 90) &
        (F.col("days_since_app_activity") <= 90)
    ) \
    .join(cleaned_customer_data, "customer_id", "left") \
    .select(
        "customer_id", "gender", "age", "loyalty_status", "city",
        "last_purchase_date", "last_app_activity",
        "days_since_purchase", "days_since_app_activity"
    ).orderBy(F.col("days_since_purchase").desc())

churn_count = churn_risk.count()
print(f"\n=== CHURN RISK: {churn_count} customers at risk ===")
print("(Haven't purchased in >90 days but active on app within 90 days)")
churn_risk.show(20, truncate=False)

# Churn risk by loyalty status
churn_by_loyalty = churn_risk.groupBy("loyalty_status") \
    .agg(F.count("*").alias("churn_risk_count")) \
    .orderBy(F.col("churn_risk_count").desc())

print("=== CHURN RISK BY LOYALTY STATUS ===")
churn_by_loyalty.show(truncate=False)

### 6. Customer 360 View — "Out of the Box" Analysis

Joins **Customer_Data** with **Clickstream** to see how many "Views" it takes for a customer to make a "Purchase", segmented by **loyalty tier** (Gold, Silver, Bronze). Also shows overall engagement metrics per tier.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CUSTOMER 360: Views Before Purchase by Loyalty Tier
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Join clickstream with customer data for loyalty status
clickstream_with_loyalty = clean_clickstream.join(
    cleaned_customer_data.select("customer_id", "loyalty_status", "gender", "age"),
    "customer_id", "left"
)

# Count views and purchases per customer per product
customer_product_funnel = clickstream_with_loyalty.groupBy("customer_id", "product_id", "loyalty_status") \
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("view_count"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("cart_count"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_count")
    )

# Filter to only journeys that ended in a purchase
purchasers = customer_product_funnel.filter(F.col("purchase_count") > 0)

# Average views before purchase by loyalty tier
views_before_purchase = purchasers.groupBy("loyalty_status") \
    .agg(
        F.round(F.avg("view_count"), 2).alias("avg_views_before_purchase"),
        F.round(F.avg("cart_count"), 2).alias("avg_cart_adds_before_purchase"),
        F.count("*").alias("total_purchase_journeys")
    ).orderBy("loyalty_status")

print("=== CUSTOMER 360: Views Before Purchase by Loyalty Tier ===")
views_before_purchase.show(truncate=False)

# Overall customer engagement summary by loyalty tier
customer_engagement = clickstream_with_loyalty.groupBy("customer_id", "loyalty_status") \
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("total_views"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("total_purchases"),
        F.countDistinct("product_id").alias("unique_products_interacted")
    )

engagement_by_loyalty = customer_engagement.groupBy("loyalty_status") \
    .agg(
        F.round(F.avg("total_events"), 2).alias("avg_events_per_customer"),
        F.round(F.avg("total_views"), 2).alias("avg_views_per_customer"),
        F.round(F.avg("total_purchases"), 2).alias("avg_purchases_per_customer"),
        F.round(F.avg("unique_products_interacted"), 2).alias("avg_products_browsed"),
        F.count("*").alias("total_customers")
    ).orderBy("loyalty_status")

print("=== CUSTOMER ENGAGEMENT BY LOYALTY TIER ===")
engagement_by_loyalty.show(truncate=False)

### 7. Cross-Channel Analysis — "Out of the Box"

Answers: *Did a customer who browsed a phone on the app (Clickstream) eventually buy it in a physical store (Sales)?* Identifies customers whose **online browsing** (Mobile/App platform) led to **in-store purchases** (channel = 'Store').

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CROSS-CHANNEL: Browsed on App → Bought in Store
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Customers who viewed products on Mobile/App platform
app_browsers = clean_clickstream \
    .filter(
        (F.col("event_type") == "view") &
        (F.col("platform").isin("Mobile", "App", "mobile", "app"))
    ) \
    .select("customer_id", "product_id").distinct()

# Customers who purchased in physical store
store_purchasers = cleaned_sales_transactions \
    .filter(F.col("channel") == "Store") \
    .select("customer_id", "product_id").distinct()

# Cross-channel: browsed on app AND bought same product in store
cross_channel = app_browsers.join(store_purchasers, ["customer_id", "product_id"], "inner") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_product_master.select("product_id", "product_name", "category", "brand"), "product_id", "left")

cross_channel_count = cross_channel.select("customer_id", "product_id").distinct().count()
unique_cross_customers = cross_channel.select("customer_id").distinct().count()

print(f"=== CROSS-CHANNEL ANALYSIS ===")
print(f"Total cross-channel instances (App Browse → Store Buy): {cross_channel_count:,}")
print(f"Unique customers with cross-channel behavior: {unique_cross_customers:,}")
cross_channel.show(20, truncate=False)

# By category
cross_by_category = cross_channel.groupBy("category") \
    .agg(F.countDistinct("customer_id").alias("unique_customers")) \
    .orderBy(F.col("unique_customers").desc())

print("=== CROSS-CHANNEL BY CATEGORY ===")
cross_by_category.show(truncate=False)

# By loyalty tier
cross_by_loyalty = cross_channel.groupBy("loyalty_status") \
    .agg(F.countDistinct("customer_id").alias("unique_customers")) \
    .orderBy(F.col("unique_customers").desc())

print("=== CROSS-CHANNEL BY LOYALTY TIER ===")
cross_by_loyalty.show(truncate=False)

### 8. Top Brands Analysis

Which brand (Apple, Samsung, etc.) is the **top seller**? Analyzes brand performance by revenue, profit, and units sold. Also shows the top 3 brands per category.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Prepare Tables to avoid "Ambiguous" errors
# Rename city/state so we know if it's the Customer's location or the Store's location
cust_df = cleaned_customer_data.withColumnRenamed("city", "cust_city").withColumnRenamed("state", "cust_state")
str_df = cleaned_store_master.withColumnRenamed("city", "store_city").withColumnRenamed("state", "store_state")

# 2. Perform the Master Join
# We drop 'category' from sales to use the one from the Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df, on="customer_id", how="left") \
    .join(str_df, on="store_id", how="left")

# 3. Calculate Profit (Ensuring all columns exist)
# Formula: Total Amount - (Quantity * Cost)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

# Verify the columns exist now
print("Verified Columns:", [c for c in gold_sales.columns if c in ["profit", "store_city", "brand"]])

In [0]:
# =====================================================
# TOP BRANDS BY REVENUE
# =====================================================
brand_performance = gold_sales.groupBy("brand") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.sum(F.abs(F.col("quantity"))).alias("total_units_sold"),
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_revenue").desc())

print("=== TOP BRANDS BY REVENUE ===")
brand_performance.show(truncate=False)

# =====================================================
# TOP 3 BRANDS PER CATEGORY (Window Function)
# =====================================================
# 1. First, get the revenue per brand per category
brand_revenue_df = gold_sales.groupBy("category", "brand") \
    .agg(F.round(F.sum("total_amount"), 2).alias("category_revenue"))

# 2. Define the window to rank brands within each category
window_spec = Window.partitionBy("category").orderBy(F.col("category_revenue").desc())

# 3. Apply the rank and filter
brand_by_category = brand_revenue_df \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("category", "rank")

print("=== TOP 3 BRANDS PER CATEGORY ===")
brand_by_category.show(50, truncate=False)

### 9. Total Revenue by City — Aggregated Gold Table

Consolidated city-level performance metrics including revenue, profit, store count, transactions, and unique customers.

In [0]:
from pyspark.sql import functions as F

# 1. Rename columns in dimension tables BEFORE the join
cust_df_clean = cleaned_customer_data \
    .withColumnRenamed("city", "cust_city") \
    .withColumnRenamed("state", "cust_state")

store_df_clean = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create gold_sales
# We drop 'category' from sales to use the one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_clean, on="customer_id", how="left") \
    .join(store_df_clean, on="store_id", how="left")

# 3. Add the 'profit' column (This solves your unresolved column error)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Success: gold_sales is now clean and has a 'profit' column.")

In [0]:
# =====================================================
# TOTAL REVENUE BY CITY (Aggregated Gold Table)
# =====================================================

total_revenue_by_city = gold_sales.groupBy("store_city", "store_state") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.countDistinct("store_id").alias("num_stores"),
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_revenue").desc())

print("=== TOTAL REVENUE BY CITY ===")
total_revenue_by_city.show(truncate=False)

Insight 1: Which Category is the "Money Maker"?
This tells the manager where to invest more marketing budget.

In [0]:
category_analysis = gold_sales.groupBy("category") \
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("profit").alias("total_profit"),
        F.count("transaction_id").alias("transaction_count")
    ).orderBy(F.col("total_profit").desc())

category_analysis.show()

Insight 2: Does Loyalty actually drive bigger Sales?
This proves if the "Gold" member program is working.

In [0]:
from pyspark.sql import functions as F

# 1. Prepare Dimension Tables to avoid overlaps
# We rename 'loyalty_status' and 'city' to make them unique
cust_df_final = cleaned_customer_data \
    .withColumnRenamed("loyalty_status", "cust_loyalty") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("state", "customer_state")

store_df_final = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create the Master Gold Table
# We drop 'category' from sales to use the official one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_final, on="customer_id", how="left") \
    .join(store_df_final, on="store_id", how="left")

# 3. Add the 'profit' column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Gold Table Rebuilt. Use 'cust_loyalty' instead of 'loyalty_status' now.")

In [0]:
# Analysis: Does Loyalty drive bigger Sales?
loyalty_analysis = gold_sales.groupBy("cust_loyalty") \
    .agg(
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit")
    ).orderBy(F.col("avg_order_value").desc())

print("=== LOYALTY STATUS PERFORMANCE ===")
loyalty_analysis.show(truncate=False)

**Insight 3: The "Return" Red Flag
Which products are being returned the most? (Using your transaction_type flag).**

In [0]:
return_analysis = gold_sales.filter(F.col("transaction_type") == "RETURN") \
    .groupBy("product_name", "category") \
    .agg(F.count("transaction_id").alias("return_count")) \
    .orderBy(F.col("return_count").desc())

print("Top 5 Most Returned Products:")
return_analysis.show(5)


### 10. Save All Gold Tables to `capstone_catalog.gold`

Persists all analytical gold tables as Delta tables for dashboarding and reporting.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. AOV and Channel Analysis ---
aov_by_channel = gold_sales.groupBy("channel").agg(
    F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue")
)

# --- 2. Conversion and Platform Analysis ---
# Calculation: (Purchases / Views) * 100
total_views = cleaned_clickstream_events.filter(F.col("event_type") == "view").count()
conversion_by_platform = cleaned_clickstream_events.groupBy("platform").agg(
    (F.count(F.when(F.col("event_type") == "purchase", 1)) / 
     F.count(F.when(F.col("event_type") == "view", 1)) * 100).alias("conversion_rate")
)

# --- 3. Stock and Risk Analysis ---
stockout_risk = cleaned_inventory_data.filter(F.col("stock_on_hand") < F.col("reorder_level"))
stockout_by_category = stockout_risk.join(cleaned_product_master, "product_id").groupBy("category").count()

# --- 4. Churn and Loyalty Analysis ---
# Customers in Clickstream but not in Sales
churn_risk = cleaned_clickstream_events.select("customer_id").distinct().join(
    cleaned_sales_transactions.select("customer_id").distinct(), "customer_id", "left_anti"
)
churn_by_loyalty = churn_risk.join(cleaned_customer_data, "customer_id").groupBy("loyalty_status").count()

# --- 5. Engagement and Cross-Channel ---
engagement_by_loyalty = cleaned_clickstream_events.join(cleaned_customer_data, "customer_id").groupBy("loyalty_status", "event_type").count()
views_before_purchase = gold_sales.groupBy("customer_id").agg(F.count("transaction_id").alias("purchase_count")) # Placeholder for Customer 360
cross_by_category = gold_sales.groupBy("category", "channel").agg(F.sum("total_amount").alias("revenue"))

In [0]:
gold_tables = {
    "gold_master_sales": gold_sales,
    "gold_category_performance": category_performance,
    "gold_city_store_performance": city_store_performance,
    "gold_aov_by_channel": aov_by_channel,
    "gold_conversion_by_platform": conversion_by_platform,
    "gold_stockout_risk": stockout_risk,
    "gold_stockout_by_category": stockout_by_category,
    "gold_churn_risk": churn_risk,
    "gold_churn_by_loyalty": churn_by_loyalty,
    "gold_views_before_purchase": views_before_purchase,
    "gold_customer_engagement_by_loyalty": engagement_by_loyalty,
    "gold_cross_channel_by_category": cross_by_category,
    "gold_brand_performance": brand_performance,
    "gold_brand_by_category": brand_by_category,
    "gold_total_revenue_by_city": total_revenue_by_city
}

for name, df in gold_tables.items():
    table_name = f"capstone_catalog.gold.{name}"
    print(f"Saving {name} → {table_name} ...")
    # Saving as Delta Tables for ACID compliance
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"Saved {name}")

A single table that tells you: "A Gold customer in a Mumbai Mall bought an Apple iPhone via the Mobile App."


In [0]:
from pyspark.sql import functions as F

# 1. Rename columns in dimension tables to avoid "Ambiguous" errors
# This ensures we know exactly which city/status we are looking at
cust_df_clean = cleaned_customer_data \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("loyalty_status", "cust_loyalty")

store_df_clean = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("store_name", "mall_name")

# 2. Create the Master Gold Table
# We join all 4 tables using their IDs
gold_master_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_clean, on="customer_id", how="left") \
    .join(store_df_clean, on="store_id", how="left")

# 3. Add a "Human Readable Insight" column
# This creates the exact sentence you asked for
gold_master_sales = gold_master_sales.withColumn(
    "transaction_story",
    F.concat(
        F.lit("A "), F.col("cust_loyalty"), F.lit(" customer in "), 
        F.col("mall_name"), F.lit(" bought an "), 
        F.col("brand"), F.lit(" "), F.col("product_name"), 
        F.lit(" via the "), F.col("channel")
    )
)

# 4. Show the result
gold_master_sales.select("transaction_id", "transaction_story").show(5, truncate=False)

In [0]:
# Filtering for the specific Mumbai/Mobile App example
mumbai_insight = gold_master_sales.filter(
    (F.col("store_city") == "Mumbai") & 
    (F.col("channel") == "Mobile App") & 
    (F.col("cust_loyalty") == "Gold")
)

mumbai_insight.select("transaction_story").show(truncate=False)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Convert your Gold Master Sales to Pandas for visualization
# We'll take a subset or aggregate to keep it performant
pdf_sales = gold_master_sales.toPandas()

# Set the visual style
sns.set_theme(style="whitegrid")

Category-wise Margin (%) Analysis
This adds the "Margin %" to your existing category analysis

In [0]:
category_margin = category_performance.withColumn(
    "margin_percentage", 
    F.round((F.col("total_profit") / F.col("total_revenue")) * 100, 2)
).orderBy(F.col("margin_percentage").desc())

category_margin.show()

Repeat vs First-Time Customer

In [0]:
# Count orders per customer
customer_order_counts = gold_sales.groupBy("customer_id") \
    .agg(F.countDistinct("transaction_id").alias("order_count"))

# Label them
customer_segments = customer_order_counts.withColumn(
    "customer_type",
    F.when(F.col("order_count") > 1, "Repeat").otherwise("First-Time")
)

customer_segments.groupBy("customer_type").count().show()

Slow-Moving SKUs (Last 30 Days)

In [0]:
# Products with 0 sales in the entire dataset
slow_moving = cleaned_product_master.join(
    cleaned_sales_transactions, "product_id", "left_anti"
).select("product_id", "product_name", "category")

print("Products with zero sales (Slow Moving):")
slow_moving.show()

VISUALIZATION


2. Visualization: Revenue by Store City
This will help identify which locations are your top performers.

In [0]:
plt.figure(figsize=(10, 6))
city_revenue = pdf_sales.groupby('store_city')['total_amount'].sum().sort_values(ascending=False)

sns.barplot(x=city_revenue.index, y=city_revenue.values, palette='viridis')

plt.title('Total Revenue by Store City', fontsize=15)
plt.xlabel('City', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)
plt.xticks(rotation=45)
plt.show()

3. Visualization: Sales Channel Performance
This chart shows whether the "Mobile App" or "In-Store" shopping is more popular

In [0]:
plt.figure(figsize=(8, 8))
channel_counts = pdf_sales['channel'].value_counts()

plt.pie(channel_counts, labels=channel_counts.index, autopct='%1.1f%%', 
        startangle=140, colors=['#66b3ff','#99ff99'])

plt.title('Distribution of Sales by Channel', fontsize=15)
plt.show()

4. Visualization: Loyalty Group Spending Habits
This helps you see if your "Gold" and "Platinum" customers are actually spending more than others

In [0]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='cust_loyalty', y='total_amount', data=pdf_sales, palette='Set2')

plt.title('Spending Distribution by Customer Loyalty Tier', fontsize=15)
plt.xlabel('Loyalty Tier', fontsize=12)
plt.ylabel('Transaction Amount ($)', fontsize=12)
plt.show()

5. Time-Series: Daily Sales Trend
This shows if your sales are growing or if there are specific "peak" days (like weekends)

In [0]:
import matplotlib.dates as mdates

# Group by date and sum revenue
daily_revenue = pdf_sales.groupby('order_date')['total_amount'].sum().reset_index()
daily_revenue['order_date'] = pd.to_datetime(daily_revenue['order_date'])
daily_revenue = daily_revenue.sort_values('order_date')

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_revenue, x='order_date', y='total_amount', marker='o', color='firebrick')

plt.title('Daily Revenue Trend (Gold Layer)', fontsize=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Sales ($)', fontsize=12)
plt.xticks(rotation=45)
plt.show()

6. Heatmap: Payment Method vs. Customer Loyalty
This visualization shows which loyalty groups prefer which payment methods (e.g., do "Platinum" members prefer "Credit Card" for points?)

In [0]:
# Create a pivot table for the heatmap
pivot_table = pdf_sales.pivot_table(index='cust_loyalty', 
                                   columns='payment_type', 
                                   values='total_amount', 
                                   aggfunc='count').fillna(0)

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_table, annot=True, fmt='g', cmap='YlGnBu')

plt.title('Heatmap: Loyalty Tier vs. Payment Preference', fontsize=15)
plt.xlabel('Payment Method', fontsize=12)
plt.ylabel('Loyalty Group', fontsize=12)
plt.show()

7. The "Top 10" Products by Quantity
In a retail capstone, you must show what is actually selling

In [0]:
plt.figure(figsize=(10, 6))
top_products = pdf_sales.groupby('product_name')['quantity'].sum().nlargest(10)

sns.barplot(x=top_products.values, y=top_products.index, palette='magma')

plt.title('Top 10 Best-Selling Products', fontsize=15)
plt.xlabel('Total Units Sold', fontsize=12)
plt.ylabel('Product Name', fontsize=12)
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# ... (your previous aggregation code here) ...
pdf_cust_segments = customer_segments.groupBy("customer_type").count().toPandas()

plt.figure(figsize=(10, 6))
# Create the barplot
ax = sns.barplot(data=pdf_cust_segments, x='customer_type', y='count', palette='viridis')

# --- THE FIX: ADD DATA LABELS ---
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points',
                fontsize=12, fontweight='bold')

plt.title('Customer Count (with exact numbers)', fontsize=15)
plt.ylabel('Count')
plt.ylim(0, pdf_cust_segments['count'].max() * 1.1) # Give some space at the top for labels
plt.show()

2. Loyalty Tier Financial Contribution (Customer Analytics)
This analysis proves which customer segment is the "heart" of the business by comparing Total Revenue vs. Total Profit across loyalty tiers (Gold, Silver, Platinum, etc.)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Calculate total quantity sold per product
product_sales = gold_master_sales.groupBy("product_name", "category") \
    .agg(F.sum("quantity").alias("total_qty_sold"))

# 2. Use Percent Rank to categorize
windowSpec = Window.orderBy(F.col("total_qty_sold").desc())
product_velocity = product_sales.withColumn("rank_percent", F.percent_rank().over(windowSpec))

inventory_velocity = product_velocity.withColumn(
    "velocity_tag",
    F.when(F.col("rank_percent") <= 0.25, "Fast Moving")
     .when(F.col("rank_percent") >= 0.75, "Slow Moving")
     .otherwise("Steady Seller")
)

# Visualization
pdf_velocity = inventory_velocity.groupBy("velocity_tag").count().toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=pdf_velocity.sort_values("count", ascending=False), 
            x='velocity_tag', y='count', palette='magma')
plt.title('Inventory Velocity: SKU Distribution', fontsize=15)
plt.ylabel('Number of Unique Products')
plt.show()

2. Loyalty Tier Financial Contribution (Customer Analytics)
This analysis proves which customer segment is the "heart" of the business by comparing Total Revenue vs. Total Profit across loyalty tiers (Gold, Silver, Platinum, etc.)

In [0]:
# Aggregate Revenue and Profit by Loyalty Tier
loyalty_finance = gold_master_sales.groupBy("cust_loyalty") \
    .agg(
        F.sum("total_amount").alias("Total_Revenue"),
        F.sum(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price"))).alias("Total_Profit")
    ).toPandas()

# Melt the dataframe for a side-by-side comparison chart
df_melted = loyalty_finance.melt(id_vars="cust_loyalty", var_name="Metric", value_name="Amount")

plt.figure(figsize=(12, 7))
sns.barplot(data=df_melted, x='cust_loyalty', y='Amount', hue='Metric', palette='Set1')

plt.title('Financial Contribution by Loyalty Tier', fontsize=15)
plt.ylabel('Amount ($)')
plt.xlabel('Loyalty Segment')
plt.legend(title='Financial Metric')
plt.show()